In [1]:
from hullwhite import GeneratePathsHWAndAssetEuler, P0T, PriceEuropeanOptionHWMC
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

## Cálculo de Gamma mediante diferencias finitas

$$
\Gamma = \frac{\partial^2 V}{\partial S_0^2}
$$

Desde el punto de vista financiero, la Gamma mide la curvatura de la relación entre el precio de la opción y el valor inicial del activo. También puede interpretarse como la sensibilidad de la Delta respecto al subyacente.

La Gamma la aproximamos mediante diferencias finitas centradas de segundo orden:

$$
\Gamma \approx \frac{V(S_0+h)-2V(S_0)+V(S_0-h)}{h^2}
$$

Usamos *common random numbers* para reducir la varianza del estimador Monte Carlo. En este caso, la misma realización de números aleatorios se utiliza para $S_0+h$, $S_0$ y $S_0-h$.

In [3]:
def GammaFD_HW_MC(NoOfPaths, NoOfSteps, T, P0T, lambd, eta, S0, sigma_S, rho, K,
                  h, option_type, seed):
    """
    Calcula la Gamma por diferencias finitas centradas
    en el modelo Hull–White + activo, usando common random numbers.
    """

    # Precio con S0 + h
    paths_up = GeneratePathsHWAndAssetEuler(
        NoOfPaths=NoOfPaths,
        NoOfSteps=NoOfSteps,
        T=T,
        P0T=P0T,
        lambd=lambd,
        eta=eta,
        S0=S0 + h,
        sigma_S=sigma_S,
        rho=rho,
        seed=seed
    )

    price_up, _ = PriceEuropeanOptionHWMC(
        paths=paths_up,
        K=K,
        T=T,
        option_type=option_type
    )

    # Precio con S0
    paths_mid = GeneratePathsHWAndAssetEuler(
        NoOfPaths=NoOfPaths,
        NoOfSteps=NoOfSteps,
        T=T,
        P0T=P0T,
        lambd=lambd,
        eta=eta,
        S0=S0,
        sigma_S=sigma_S,
        rho=rho,
        seed=seed
    )

    price_mid, _ = PriceEuropeanOptionHWMC(
        paths=paths_mid,
        K=K,
        T=T,
        option_type=option_type
    )

    # Precio con S0 - h
    paths_down = GeneratePathsHWAndAssetEuler(
        NoOfPaths=NoOfPaths,
        NoOfSteps=NoOfSteps,
        T=T,
        P0T=P0T,
        lambd=lambd,
        eta=eta,
        S0=S0 - h,
        sigma_S=sigma_S,
        rho=rho,
        seed=seed
    )

    price_down, _ = PriceEuropeanOptionHWMC(
        paths=paths_down,
        K=K,
        T=T,
        option_type=option_type
    )

    gamma_fd = (price_up - 2.0 * price_mid + price_down) / (h**2)

    return gamma_fd, price_up, price_mid, price_down

In [4]:
# Parámetros del modelo
NoOfPaths = 20000
NoOfSteps = 250
T = 1.0

lambd = 0.5
eta = 0.01

# Parámetros del activo
S0 = 100.0
sigma_S = 0.20
rho = -0.3

# Parámetros de la opción
K = 100.0
option_type = "call"

# Tamaño de perturbación
h = 0.5

gamma_fd, price_up, price_mid, price_down = GammaFD_HW_MC(
    NoOfPaths=NoOfPaths,
    NoOfSteps=NoOfSteps,
    T=T,
    P0T=P0T,
    lambd=lambd,
    eta=eta,
    S0=S0,
    sigma_S=sigma_S,
    rho=rho,
    K=K,
    h=h,
    option_type=option_type,
    seed=123
)

print(f"Precio con S0 + h = {price_up:.6f}")
print(f"Precio con S0     = {price_mid:.6f}")
print(f"Precio con S0 - h = {price_down:.6f}")
print(f"Gamma FD          = {gamma_fd:.6f}")

Precio con S0 + h = 10.704435
Precio con S0     = 10.384148
Precio con S0 - h = 10.068360
Gamma FD          = 0.017995


#### Resultado numérico para Gamma por diferencias finitas

El valor obtenido para la Gamma es positivo, como se espera en el caso de una opción call europea vanilla. Esto refleja la convexidad de la relación entre el precio de la opción y el precio inicial del activo.

Desde el punto de vista financiero, una Gamma positiva indica que la Delta aumenta cuando el precio del subyacente sube. En este caso, una Gamma cercana a $0.018$ nos da una curvatura moderada.
